In [ ]:
from pyspark.sql.functions import col

In [ ]:
## Variables

gold_lakehouse = "Metrics_Lakehouse"
gold_table_name = "timepoint_metrics"
capacity_table = "capacities"

start_date = "01 May 2026 00:00"
end_date = "10 May 2026 00:00"

#verbose mode displaying data
display_data = True

count_of_connectivity_errors = 0

#### Add Regional Pricing

In [ ]:
columns = ["Region", "CU_price_per_hour", "Type", "currency"]
data = [("North Europe", 0.380, "PAYG", "USD")
        ,("North Europe", 0.227, "Reservation", "USD")
        ,("West Europe", 0.440, "PAYG", "USD")
        ,("West Europe", 0.262, "Reservation", "USD")
        ,("West US 3", 0.360, "PAYG", "USD")
        ,("West US 3", 0.215, "Reservation", "USD")
        ]
capacity_pricing = spark.createDataFrame(data, schema=columns)

if display_data:
    display(capacity_pricing)


#### Check default region and fix so that mapping works

In [ ]:
#fetch list of capacity regions, deal with default region
query = f"SELECT CapacityId, case Region when 'Default' then 'West US 3' else Region end  as Region FROM {gold_lakehouse}.{capacity_table}"
capacity_mapping = spark.sql(query)
if display_data:
    display(capacity_mapping)

#### Choose to see PAYG or Reservation pricing

In [ ]:
# Join capacity_mapping with capacity_pricing on Region
capacity_with_pricing = capacity_mapping.join(
    capacity_pricing,
    on="Region",
    how="left"
)

# Subset to show only Type = 'Reservation' or 'PAYG'
capacity_with_pricing = capacity_with_pricing.filter(capacity_with_pricing.Type == "Reservation")

if display_data:
    display(capacity_with_pricing)


In [ ]:
#Check for kinds of Operations
query = f"""
        select Operation, count(*) as RecCount from {gold_lakehouse}.{gold_table_name} 
        where TimePoint BETWEEN to_timestamp('{start_date}', 'dd MMMM yyyy HH:mm')
        AND to_timestamp('{end_date}', 'dd MMMM yyyy HH:mm')
        group by Operation
        """


operation_types = spark.sql(query)

display(operation_types)

#### Fetch activity data and filter for dates or Operation Type 

There will be duplicates for operations by timepoint, deal with these with a windowing function and status <BR>

Add additional operation types for more exploration

In [ ]:
query = f"""
        select Workspace_name, Item_name, Unique_key, Operation, Operation_Id, Operation_start_time, Operation_end_time
        , Status, SumDuration_s as Duration, SumTotal_CU_s as Total_CU_s, Timepoint, CapacityID
        from
        (
        select * ,
        ROW_NUMBER() OVER (
            PARTITION BY Operation_start_time, Operation_end_time
            ORDER BY Timepoint DESC
        ) AS rownum
        from {gold_lakehouse}.{gold_table_name} 
        where Operation = 'Notebook Run' and Status = 'Stopped'
        and TimePoint BETWEEN to_timestamp('{start_date}', 'dd MMMM yyyy HH:mm')
        AND to_timestamp('{end_date}', 'dd MMMM yyyy HH:mm')
        )
        where rownum = 1
        """
                        
print(f"Fetching data with this query: {query}")


activities = spark.sql(query)
if display_data:
    display(activities)

In [ ]:
# Join activities with capacity_with_pricing on CapacityId

activities_with_pricing = activities.join(
    capacity_with_pricing.select("CapacityId", "CU_price_per_hour", "Type", "currency"),
    on="CapacityId",
    how="left"
)

# Add cost column: (sumtotalc_cu_s * CU_price_per_hour) / 3600
activities_with_pricing = activities_with_pricing.withColumn(
    "Cost",
    (col("Total_CU_s") / 3600 * col("CU_price_per_hour")) 
)

# calculate the cost of each activity
if display_data:
    display(activities_with_pricing)


In [ ]:
# Summarise pricing into a new DataFrame
cost_summary_df = activities_with_pricing.select(
    "Workspace_name",
    "Item_name",
    "Operation",
    "Operation_start_time",
    "Operation_end_time",
    "Workspace_name",
    "Duration",
    "Total_CU_s",
    "Cost"
)

if display_data:
    display(cost_summary_df)
